# Update Agent URDB Rate Mapping

Updates `tariff_dict` in the runs pickle (`agent_df_base_res_national_updated_wholesale_prices.pkl`)
to use current URDB rates, by following the `supersedes` chain from each agent's label.

The pickle doesn't have a `label` column, so labels are joined from `residential_agents_updated.csv`
on `bldg_id`. The updated `tariff_dict` is stored alongside the original for comparison before
writing back.

In [1]:
import json
import pandas as pd
import os
import sys

DATA_DIR      = "../../../data"
INPUT_AGENTS  = "../../../dgen_os/input_agents"
PICKLE_PATH         = os.path.join(INPUT_AGENTS, "agent_df_base_res_national_updated_wholesale_prices.pkl")
PICKLE_PATH_UPDATED = os.path.join(INPUT_AGENTS, "agent_df_base_res_national_updated_tariffs_2026.pkl")
LABEL_CSV     = os.path.join(DATA_DIR, "residential_agents_updated.csv")
URDB_CSV_PATH = os.path.join(DATA_DIR, "usurdb.csv")
URDB_JSON_PATH= os.path.join(DATA_DIR, "usurdb.json")

sys.path.insert(0, '../')
from urdb_to_pysam import urdb_to_pysam

In [3]:
# Load pickle (the file actually used in dGen runs)
# agent_id is the index — reset it to a column so it survives the merge
agents = pd.read_pickle(PICKLE_PATH).reset_index()

# Join label from CSV on bldg_id (pickle doesn't have label column)
label_df = pd.read_csv(LABEL_CSV, low_memory=False, usecols=["bldg_id", "label"])
agents = agents.merge(label_df, on="bldg_id", how="left")

# Load URDB
urdb_csv = pd.read_csv(URDB_CSV_PATH, low_memory=False, usecols=["label", "supersedes"])
with open(URDB_JSON_PATH) as f:
    urdb_json = json.load(f)
urdb_by_label = {r["_id"]["$oid"]: r for r in urdb_json}

print(f"Pickle rows:            {len(agents):,}")
print(f"Labels joined:          {agents['label'].notna().sum():,}")
print(f"Unique agent labels:    {agents['label'].nunique():,}")
print(f"URDB JSON records:      {len(urdb_by_label):,}")
print(f"agent_id in columns:    {'agent_id' in agents.columns}")

Pickle rows:            24,660
Labels joined:          24,163
Unique agent labels:    977
URDB JSON records:      63,244
agent_id in columns:    True


In [4]:
# Build a FORWARD map: old_label -> new_label
# The 'supersedes' field is a backward pointer (new rate points to old rate it replaced).
# We reverse it so we can follow forward from an old label to the newest one.
superseded_by = (
    urdb_csv.dropna(subset=["supersedes"])
    .set_index("supersedes")["label"]
    .to_dict()
)

print(f"Superseded-by map entries: {len(superseded_by):,}")

Superseded-by map entries: 25,517


In [5]:
def follow_to_latest(label, max_depth=50):
    """Follow the forward supersedes chain to the most up-to-date label."""
    if pd.isna(label):
        return None
    visited, current = set(), label
    for _ in range(max_depth):
        if current in visited:
            break
        visited.add(current)
        nxt = superseded_by.get(current)
        if nxt is None:
            break
        current = nxt
    return current

unique_labels = agents["label"].dropna().unique()
resolved = {lbl: follow_to_latest(lbl) for lbl in unique_labels}

changed = {k: v for k, v in resolved.items() if k != v}
print(f"Labels resolved:                   {len(resolved):,}")
print(f"Labels with a newer version found: {len(changed):,}")
print()
for old, new in list(changed.items())[:10]:
    print(f"  {old}  ->  {new}")

Labels resolved:                   977
Labels with a newer version found: 367

  539fb49cec4f024bc1dbeb73  ->  539fb810ec4f024bc1dc12af
  539f72b0ec4f024411ecf411  ->  539fba55ec4f024bc1dc2c69
  568da4c35457a3d82eabca33  ->  69a99f187cef8411ae01894b
  539fb846ec4f024bc1dc1517  ->  53a072eb5257a39a6f04ca0f
  5cacc7715457a393487780e2  ->  69a706ab7da20ea6b00d0238
  539fca4dec4f024d2f53fa1e  ->  6549645e52e4d31bea0f4e7a
  56c77d785457a3410cb338bb  ->  69a1b8bf40140c3bb007f1bd
  539fc83dec4f024d2f53e46e  ->  5405d1ed5257a3d222af34b1
  539fb86aec4f024bc1dc1713  ->  6509f9bafc971086940db86d
  539fb6a4ec4f024bc1dc028f  ->  69a1bef91c767657640eb08e


## Manual Overrides

For utilities where the supersedes chain is broken (terminates at an old rate with an end date),
we manually specify the correct current label below. Each entry includes a note explaining why
the manual mapping is needed.

In [6]:
# Manual label overrides: agent_label -> (updated_label, note)
# Used when the supersedes chain is broken or connects to the wrong rate product.
#
# Rate selection guidance:
#   - Prefer serviceType = "Delivery with Standard Offer" over "Delivery" only
#   - Use single-phase rates for residential (not three-phase)
#   - Match the rate name/product to what the agent file originally had
MANUAL_OVERRIDES = {
    # Florida Power & Light RS-1 Residential Service
    # Chain breaks in 2015; manually mapped to bundled RS-1 rate effective 2026-01-01
    "53a455f05257a3ff4a8d8cf2": ("69a9c4fc2f2357589100972d", "FPL RS-1 bundled rate, effective 2026-01-01"),

    # Baltimore Gas & Electric - Residential Service (Schedule R)
    # Chain terminates at 2017; correct current Schedule R rate effective 2026-01-01
    "574382c25457a33d0c906f5b": ("69a728bdd1ab4183610bb29a", "BGE Schedule R, effective 2026-01-01"),

    # Detroit Edison (DTE) - D-1 Residential Service (Full)
    # Agent label not in URDB CSV; manually mapped to current D1 full service rate effective 2026-04-01
    "5b8437755457a3f618edfe00": ("69e681d00b9e1bb34f0912d9", "DTE Detroit Edison D-1 Full Service, effective 2026-04-01"),

    # PECO Energy - Residential Service (R)
    # Chain breaks in 2018; manually mapped to current R rate effective 2026-01-01
    "5550e4025457a3107e8b4568": ("69e65026bc32447e430e25a9", "PECO Residential Service R, effective 2026-01-01"),

    # Ohio Power Co - Residential Service Ohio Power Rate Zone (RS)
    # Agent label not in URDB CSV; mapped to bundled RS rate effective 2026-04-10
    "5bb50cfe5457a3f55e68a129": ("69dd1b5381bf8b311c0576a8", "Ohio Power Bundled RS, effective 2026-04-10"),

    # Wisconsin Electric - Residential and Farm Rates Single Phase (Rg1)
    # Agent label not traceable; mapped to current single-phase Rg1 rate effective 2025-05-31
    "5bbbc9a45457a38c4f135b4d": ("6859a625ee417fe3fa0cb0df", "Wisconsin Electric Residential Single-Phase Rg1, effective 2025-05-31"),

    # Massachusetts Electric Co - R-1 Residential
    # Chain breaks in 2016; mapped to current R-1 rate effective 2025-03-01
    "539fc3b3ec4f024c27d8bf3f": ("67cf3302ef8f02942e0586be", "Massachusetts Electric R-1 Residential, effective 2025-03-01"),

    # Long Island Power Authority (LIPA) - 180/183/186 Residential Service
    # Chain breaks at 2016; mapped to Delivery with Standard Offer rate effective 2026-01-01
    "539f74b2ec4f024411ed0b69": ("6973ea390a32312ef00ae241", "LIPA 180/183/186 Residential Delivery with Standard Offer, effective 2026-01-01"),

    # NY Orange & Rockland - WEST SERVICE CLASSIFICATION NO. 1 Residential NSS
    # Label not in CSV; mapped to Delivery with Standard Offer rate effective 2025-02-26
    "576972975457a34d4dd4b921": ("67bf2b23defa2cd03006bfb3", "O&R West SC1 Residential NSS Delivery with Standard Offer, effective 2025-02-26"),

    # Pennsylvania Electric Co - Residential TOU Service (NY territory)
    # Chain stops at 2016; no NY-specific rate in URDB; using PA Residential Time-of-Day
    # Delivery with Standard Offer rate effective 2025-08-01 as best available match
    "539f6d09ec4f024411ecb0cb": ("68a58f6e1e8e03bc7807708c", "Penelec Residential Time-of-Day Delivery with Standard Offer, effective 2025-08-01"),

    # Duquesne Light Co - Residential Service
    # Mapped to Delivery with Standard Offer rate effective 2025-09-01
    "539fcad6ec4f024d2f53ffde": ("692ba5ba05a176f84707caba", "Duquesne Light Residential Service Delivery with Standard Offer, effective 2025-09-01"),

    # Metropolitan Edison Co (Pennsylvania) - Residential Service Rate
    # Old rate was delivery-only; mapped to Delivery with Standard Offer rate effective 2025-07-01
    "539fbcadec4f024c27d872d9": ("688c63f3f9e318b107054d0a", "Met Ed Residential Service Rate Delivery with Standard Offer, effective 2025-07-01"),

    # City of San Antonio / CPS Energy - RE - Residential Service
    # Chain breaks in 2014; mapped to current RE rate effective 2026-01-01
    "539fb56dec4f024bc1dbf513": ("69ea5c79d944dd4e3c095d40", "CPS Energy RE Residential Service, effective 2026-01-01"),

    # El Paso Electric Co - RS (Residential Service)
    # Chain breaks in 2016; mapped to current bundled Residential Service effective 2025-01-01
    "539f6e1bec4f024411ecbdff": ("699c81cba137818a0d03e869", "El Paso Electric Residential Service Bundled, effective 2025-01-01"),

    # Entergy Louisiana Inc (formerly Entergy Gulf States Louisiana) - Residential and Farm Service RS-L
    # Utility rebranded; mapped to current Residential Service (RS) effective 2026-03-02 (applies to LA and AR)
    "568da4c35457a3d82eabca33": ("69a99f187cef8411ae01894b", "Entergy Louisiana Residential Service RS Bundled, effective 2026-03-02"),

    # Alabama Power Co - Family Dwelling Service
    # Chain orphaned; mapped to current Family Dwelling Service effective 2026-01-01
    "53c3fb575257a3d37e5b36b6": ("69deb3d8e547dde41109ee02", "Alabama Power Family Dwelling Service Bundled, effective 2026-01-01"),

    # Central Maine Power Co - A Residential Standard Offer Service (Bundled)
    # Mapped to current bundled rate effective 2026-01-01
    "539fb9d8ec4f024bc1dc2739": ("6993c9d33444dea9740958a9", "Central Maine Power Residential Standard Offer Bundled, effective 2026-01-01"),

    # Interstate Power and Light Co - Optional Residential Service (IA + MN)
    # Mapped to current Electric Residential Service Usage - 400, effective 2025-07-01
    "539fc15cec4f024c27d8a5d7": ("688099521ed11411230c3a2d", "IPL Electric Residential Service Usage-400 Bundled, effective 2025-07-01"),

    # Interstate Power and Light Co - Optional Residential Time of Use (IA)
    # Mapped to current Electric Residential Service Usage Time of Day - 407, effective 2025-07-01
    "539fc320ec4f024c27d8b88f": ("6880a3bca0fd88bf8a0866fd", "IPL Electric Residential Service Usage TOU-407 Bundled, effective 2025-07-01"),

    # Duke Energy Ohio - RATE RS Residential Service
    # Previous label expired 2024-02-29; mapped to current rate effective 2026-01-02
    "539f729fec4f024411ecf31f": ("6971698df015160d35029bad", "Duke Energy Ohio RATE RS Delivery with Standard Offer, effective 2026-01-02"),

    # Virginia Electric & Power Co (Dominion) - Residential Schedule 1
    # Applies to VA and NC agents; mapped to current bundled rate effective 2026-01-01
    "539fc112ec4f024c27d8a2d3": ("6997c46543b6aa9b2a015f78", "Dominion Virginia Electric Residential Schedule 1 Bundled, effective 2026-01-01"),

    # Ameren Illinois Company - DS-1 Residential Zone 1
    # Mapped to current Delivery with Standard Offer rate effective 2026-01-01
    "579fd7325457a3d77b38a22c": ("69cad0f1689c75b85803a0f8", "Ameren Illinois DS-1 Residential Zone 1 Delivery with Standard Offer, effective 2026-01-01"),

    # Atlantic City Electric Co (NJ) - RS Delivery Service
    # Chain orphaned; mapped to current RS rate effective 2025-07-01
    "539f735aec4f024411ecfbe9": ("68a0502918cc45e4980c6e91", "Atlantic City Electric RS Delivery with Standard Offer, effective 2025-07-01"),

    # Rochester Gas & Electric Corp - SC1 Residential Service RSS
    # Mapped to current Delivery with Standard Offer rate effective 2025-05-01
    "59b99f025457a3943e42da65": ("68c87662584527c3070c367d", "Rochester Gas & Electric SC1 RSS Delivery with Standard Offer, effective 2025-05-01"),
}

# Labels to skip entirely — pickle already has a correct current rate applied by a previous researcher.
# The supersedes chain for these terminates at an old end-dated rate which would overwrite a better value.
SKIP_LABELS = {
    "539f747dec4f024411ed0927",  # PEPCO Residential - Schedule R: pickle has $0.234/kWh flat (current)
}

def resolve_label(label):
    """Return the best available updated label, applying manual overrides first."""
    if pd.isna(label):
        return None
    if label in SKIP_LABELS:
        return None  # explicitly skip — leave original tariff_dict intact
    if label in MANUAL_OVERRIDES:
        return MANUAL_OVERRIDES[label][0]
    return follow_to_latest(label)

print(f"Manual overrides defined: {len(MANUAL_OVERRIDES)}")
print(f"Skip labels: {len(SKIP_LABELS)}")
for agent_lbl, (updated_lbl, note) in MANUAL_OVERRIDES.items():
    print(f"  {agent_lbl} -> {updated_lbl}  # {note}")

Manual overrides defined: 24
Skip labels: 1
  53a455f05257a3ff4a8d8cf2 -> 69a9c4fc2f2357589100972d  # FPL RS-1 bundled rate, effective 2026-01-01
  574382c25457a33d0c906f5b -> 69a728bdd1ab4183610bb29a  # BGE Schedule R, effective 2026-01-01
  5b8437755457a3f618edfe00 -> 69e681d00b9e1bb34f0912d9  # DTE Detroit Edison D-1 Full Service, effective 2026-04-01
  5550e4025457a3107e8b4568 -> 69e65026bc32447e430e25a9  # PECO Residential Service R, effective 2026-01-01
  5bb50cfe5457a3f55e68a129 -> 69dd1b5381bf8b311c0576a8  # Ohio Power Bundled RS, effective 2026-04-10
  5bbbc9a45457a38c4f135b4d -> 6859a625ee417fe3fa0cb0df  # Wisconsin Electric Residential Single-Phase Rg1, effective 2025-05-31
  539fc3b3ec4f024c27d8bf3f -> 67cf3302ef8f02942e0586be  # Massachusetts Electric R-1 Residential, effective 2025-03-01
  539f74b2ec4f024411ed0b69 -> 6973ea390a32312ef00ae241  # LIPA 180/183/186 Residential Delivery with Standard Offer, effective 2026-01-01
  576972975457a34d4dd4b921 -> 67bf2b23defa2cd0300

## Synthetic Tariffs

For deregulated ERCOT utilities where no usable bundled residential rate exists in URDB,
we construct synthetic tariff_dicts directly from authoritative rate components:
- **Delivery**: PUCT TDU Rate Summary (effective June 1, 2026)
- **Supply**: ~9.0¢/kWh mid-market ERCOT residential fixed-rate assumption (PowerToChoose / ElectricChoice, May–June 2026)

These are keyed by `eia_id` and applied to agents with no label.

In [7]:
def _flat_tariff_dict(fixed_monthly, rate_per_kwh):
    """Build a minimal PySAM tariff_dict for a flat-rate residential plan."""
    return {
        "en_electricity_rates":    1,
        "ur_metering_option":      0,
        "ur_monthly_fixed_charge": fixed_monthly,
        "ur_ec_sched_weekday":     [[1]*24 for _ in range(12)],
        "ur_ec_sched_weekend":     [[1]*24 for _ in range(12)],
        "ur_ec_tou_mat":           [[1.0, 1.0, 1e38, 0.0, rate_per_kwh, 0.0]],
        "ur_dc_enable":            0,
        "ur_enable_billing_demand": False,
    }

# Synthetic tariffs keyed by (eia_id, tariff_name) — only applied to exact matches
# Delivery rates from PUCT TDU Rate Summary effective 2026-06-01
# Supply: 9.0¢/kWh mid-market ERCOT assumption (PowerToChoose/ElectricChoice May-Jun 2026)
SYNTHETIC_TARIFFS = {
    # CenterPoint Energy Houston — delivery 5.1461¢ + supply 9.0¢ = 14.15¢/kWh
    # Fixed: $2.11 customer + $2.79 metering = $4.90/month
    (8901, "Residential"): _flat_tariff_dict(fixed_monthly=4.90, rate_per_kwh=0.1415),

    # AEP Texas Central — delivery 5.9¢ + supply 9.0¢ = 14.9¢/kWh
    # Fixed: $1.27 customer + $1.97 metering = $3.24/month
    (3278, "Residential"): _flat_tariff_dict(fixed_monthly=3.24, rate_per_kwh=0.149),
}

print(f"Synthetic tariffs defined: {len(SYNTHETIC_TARIFFS)}")
for (eia, tariff), td in SYNTHETIC_TARIFFS.items():
    rate = td['ur_ec_tou_mat'][0][4]
    fixed = td['ur_monthly_fixed_charge']
    print(f"  eia_id={eia}, tariff_name='{tariff}' -> ${fixed:.2f}/mo + ${rate:.4f}/kWh")

Synthetic tariffs defined: 2
  eia_id=8901, tariff_name='Residential' -> $4.90/mo + $0.1415/kWh
  eia_id=3278, tariff_name='Residential' -> $3.24/mo + $0.1490/kWh


In [8]:
# Preserve original tariff_dict before anything is overwritten
agents["tariff_dict_original"] = agents["tariff_dict"]

# Resolve updated labels for all agents
agents["updated_label"] = agents["label"].apply(resolve_label)

def make_updated_tariff_dict(row):
    """Return updated tariff_dict from URDB chain/override, or synthetic if applicable."""
    # 1. Try URDB-based update via label
    updated_label = row["updated_label"]
    if pd.notna(updated_label):
        record = urdb_by_label.get(updated_label)
        if record is not None:
            return urdb_to_pysam(record)

    # 2. Fall back to synthetic tariff keyed by (eia_id, tariff_name)
    key = (int(row["eia_id"]), row["tariff_name"]) if pd.notna(row.get("eia_id")) else None
    if key and key in SYNTHETIC_TARIFFS:
        return SYNTHETIC_TARIFFS[key]

    return None

agents["tariff_dict_updated"] = agents.apply(make_updated_tariff_dict, axis=1)

# Summary
n_total      = len(agents)
n_has_label  = agents["label"].notna().sum()
n_updated    = agents["tariff_dict_updated"].notna().sum()
n_unchanged  = (agents["label"] == agents["updated_label"]).sum()
n_synthetic  = agents.apply(
    lambda r: (int(r["eia_id"]), r["tariff_name"]) in SYNTHETIC_TARIFFS
              if pd.notna(r.get("eia_id")) else False, axis=1).sum()

print(f"Total agents in pickle:            {n_total:,}")
print(f"Agents with a label (joined):      {n_has_label:,}")
print(f"Got updated tariff_dict:           {n_updated:,} ({100*n_updated/n_total:.1f}%)")
print(f"  of which synthetic:              {n_synthetic:,}")
print(f"Label unchanged (already latest):  {n_unchanged:,}")

# Spot-check NJ bldg_id 13695
row = agents[agents["bldg_id"] == 13695].iloc[0]
print(f"\nSpot-check bldg_id 13695 (NJ PSE&G):")
print(f"  original label:          {row['label']}")
print(f"  updated label:           {row['updated_label']}")
print(f"  tariff_dict_updated keys: {list(row['tariff_dict_updated'].keys()) if row['tariff_dict_updated'] else 'None'}")

# Spot-check CenterPoint
cp = agents[(agents["eia_id"].astype(str) == "8901") & (agents["tariff_name"] == "Residential")].iloc[0]
print(f"\nSpot-check CenterPoint (eia_id=8901, Residential):")
print(f"  tariff_dict_updated: {cp['tariff_dict_updated']}")

Total agents in pickle:            24,660
Agents with a label (joined):      24,163
Got updated tariff_dict:           24,407 (99.0%)
  of which synthetic:              347
Label unchanged (already latest):  5,047

Spot-check bldg_id 13695 (NJ PSE&G):
  original label:          56f066205457a3d35b7112bc
  updated label:           678a981bdad1879587025dc0
  tariff_dict_updated keys: ['en_electricity_rates', 'ur_metering_option', 'ur_monthly_fixed_charge', 'ur_ec_sched_weekday', 'ur_ec_sched_weekend', 'ur_ec_tou_mat', 'ur_dc_flat_mat', 'ur_dc_tou_mat', 'ur_dc_sched_weekday', 'ur_dc_sched_weekend', 'ur_dc_enable', 'ur_enable_billing_demand']

Spot-check CenterPoint (eia_id=8901, Residential):
  tariff_dict_updated: {'en_electricity_rates': 1, 'ur_metering_option': 0, 'ur_monthly_fixed_charge': 4.9, 'ur_ec_sched_weekday': [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 

### Notes on specific utilities

**Tennessee / Kingsport Power Co:** The supersedes chain for Kingsport's "Residential Electric Service"
is intact and terminates at a 2025 rate, but the effective rate shows a ~21% decrease from the agent
file to the current rate. This is suspicious given that most utilities have seen rate increases.
A likely explanation is that many Tennessee households in the agent file are incorrectly categorized
as Kingsport Power Co customers when they are actually served by TVA or other Tennessee utilities.
No manual adjustment is being made at this time pending further investigation.

## Rate Comparison Table

Compare old (pickle) vs new (updated URDB) effective rates by utility, weighted by households.
Uses `avg_monthly_kwh` per agent for the bill simulation.

In [9]:
import sys
sys.path.insert(0, '../')
from financial_functions import normalize_tariff
from collections import defaultdict

def effective_rate(tariff_dict, monthly_kwh=900.0):
    """Simulate bill at monthly_kwh and return effective $/kWh."""
    try:
        td = normalize_tariff(tariff_dict)
    except Exception:
        return None
    mat   = td.get("ur_ec_tou_mat") or []
    sched = td.get("ur_ec_sched_weekday") or []
    if not mat:
        return None
    month_counts = {}
    for row in sched:
        p = row[0] if row else 1
        month_counts[p] = month_counts.get(p, 0) + 1
    by_period = defaultdict(list)
    for row in mat:
        by_period[int(row[0])].append(row)
    total_bill, total_months = 0.0, sum(month_counts.values()) or 12
    for period, rows in by_period.items():
        rows_sorted = sorted(rows, key=lambda r: r[1])
        remaining, bill, prev_max = monthly_kwh, 0.0, 0.0
        for row in rows_sorted:
            tier_cap = max(min(float(row[2]), monthly_kwh) - prev_max, 0.0)
            kwh_in   = min(remaining, tier_cap)
            bill    += kwh_in * float(row[4])
            remaining -= kwh_in
            prev_max  = min(float(row[2]), monthly_kwh)
            if remaining <= 0:
                break
        if remaining > 0 and rows_sorted:
            bill += remaining * float(rows_sorted[-1][4])
        total_bill += bill * month_counts.get(period, 1)
    return (total_bill / total_months) / monthly_kwh

def agent_kwh(row):
    v = row.get("avg_monthly_kwh")
    return float(v) if v and float(v) > 10 else 900.0

# Utility name from label
urdb_full = pd.read_csv(URDB_CSV_PATH, low_memory=False, usecols=["label", "utility", "eiaid"])
label_utility = urdb_full.dropna(subset=["label", "utility"]).set_index("label")["utility"].to_dict()
eia_utility = (urdb_full.dropna(subset=["eiaid", "utility"])
               .groupby("eiaid")["utility"].agg(lambda x: x.mode()[0])
               .rename(index=lambda k: str(int(k)))
               .to_dict())
agents["utility_name"] = agents["label"].map(label_utility).fillna(
    agents["eia_id"].astype(str).map(eia_utility)
)

# Use tariff_dict_original so this is unaffected by the write cell
def old_rate(row):
    try:
        return effective_rate(row["tariff_dict_original"], agent_kwh(row))
    except: return None

def new_rate(row):
    try:
        td = row["tariff_dict_updated"]
        if not td: return None
        return effective_rate(td, agent_kwh(row))
    except: return None

agents["_old_rate"] = agents.apply(old_rate, axis=1)
agents["_new_rate"] = agents.apply(new_rate, axis=1)

def wavg(g, col):
    w, v = g["customers_in_bin_initial"], g[col]
    mask = v.notna() & w.notna() & (w > 0)
    if not mask.any(): return None
    return (v[mask] * w[mask]).sum() / w[mask].sum()

comp = (
    agents.groupby(["state_abbr", "utility_name", "tariff_name"], dropna=False)
    .apply(lambda g: pd.Series({
        "total_households": g["customers_in_bin_initial"].sum(),
        "old_rate":         wavg(g, "_old_rate"),
        "new_rate":         wavg(g, "_new_rate"),
    }), include_groups=False)
    .reset_index()
    .sort_values("total_households", ascending=False)
)

total = comp["total_households"].sum()
comp["pct_hh"]         = (100 * comp["total_households"] / total).round(1)
comp["cumul_pct"]      = comp["pct_hh"].cumsum().round(1)
comp["old_rate_$/kWh"] = comp["old_rate"].apply(lambda x: f"${x:.3f}" if pd.notna(x) else "—")
comp["new_rate_$/kWh"] = comp["new_rate"].apply(lambda x: f"${x:.3f}" if pd.notna(x) else "—")
comp["change"]         = comp.apply(
    lambda r: f"+{(r.new_rate/r.old_rate - 1)*100:.0f}%"
    if pd.notna(r.old_rate) and pd.notna(r.new_rate) and r.old_rate > 0 else "—", axis=1)
comp["total_households"] = comp["total_households"].round(0).astype(int)

display_cols = ["state_abbr", "utility_name", "tariff_name", "total_households",
                "pct_hh", "cumul_pct", "old_rate_$/kWh", "new_rate_$/kWh", "change"]

out = comp[display_cols][comp["cumul_pct"] <= 80]
out_path = os.path.join(DATA_DIR, "urdb_rate_comparison.csv")
out.to_csv(out_path, index=False)
print(f"Saved {len(out)} rows to {out_path}")
out.head(40)

Saved 106 rows to ../../../data/urdb_rate_comparison.csv


,state_abbr,utility_name,tariff_name,total_households,pct_hh,cumul_pct,old_rate_$/kWh,new_rate_$/kWh,change
908,TX,Entergy Texas Inc.,Residential Service - Time Of Day,5817172,4.8,4.8,$0.091,$0.132,+46%
70,CA,Pacific Gas & Electric Co,E-1 -Residential Service Baseline Region P,4949319,4.1,8.9,$0.362,$0.492,+36%
147,FL,Florida Power & Light Co.,RS-1 Residential Service,4888060,4.0,12.9,$0.094,$0.124,+33%
77,CA,Southern California Edison Co,Time-of-use Tiered Domestic (NEM 2.0): TOU-D-B,4700316,3.9,16.8,$0.204,$0.306,+50%
708,NY,Consolidated Edison Co-NY Inc,SC-1 - Residential & Religious Service [NYC],3179314,2.6,19.4,$0.135,$0.345,+157%
263,IL,Commonwealth Edison Co,BES - Residential Single Family Without Electr...,3127980,2.6,22.0,$0.103,$0.126,+23%
887,TN,Kingsport Power Co (Tennessee),Residential Electric Service,2556686,2.1,24.1,$0.102,$0.081,+-21%
963,VA,Virginia Electric & Power Co (North Carolina),Residential Schedule 1,2386264,2.0,26.1,$0.097,$0.162,+67%
603,NC,Progress Energy Carolinas Inc (South Carolina),Residential Service (RES-41) Single Phase,2309912,1.9,28.0,$0.104,$0.135,+30%
899,TX,CenterPoint Energy,Residential,2256546,1.9,29.9,$0.094,$0.141,+51%


In [ ]:
# ============================================================
# WRITE UPDATED PICKLE — run only when satisfied with comparison
# ============================================================
# import json

# mask = agents["tariff_dict_updated"].notna()
# agents.loc[mask, "tariff_dict"] = agents.loc[mask, "tariff_dict_updated"].apply(json.dumps)
# agents_out = agents.drop(columns=["label", "updated_label", "tariff_dict_original", "tariff_dict_updated"])

# # Restore agent_id as the index (as it was in the original pickle)
# agents_out = agents_out.set_index("agent_id")

# print(f"Agents with updated tariff_dict: {mask.sum():,} / {len(agents):,}")
# print(f"Output columns: {agents_out.shape[1]}")
# print(f"Index name: {agents_out.index.name}")
# agents_out.to_pickle(PICKLE_PATH_UPDATED)  # saves as NEW file — original is preserved
# print(f"\nSaved to {PICKLE_PATH_UPDATED}")
# print(f"Next: gsutil cp {PICKLE_PATH_UPDATED} gs://dgen-assets/input_agents/agent_df_base_res_national_updated_tariffs_2026.pkl")

Agents with updated tariff_dict: 24,407 / 24,660
Output columns: 45
Index name: agent_id

Saved to ../../../dgen_os/input_agents/agent_df_base_res_national_updated_tariffs_2026.pkl
Next: gsutil cp ../../../dgen_os/input_agents/agent_df_base_res_national_updated_tariffs_2026.pkl gs://dgen-assets/input_agents/agent_df_base_res_national_updated_tariffs_2026.pkl


## State-Level Average Price Check

Replicates the `avg_price_2026_model` metric — weighted average effective $/kWh by state,
using updated tariff_dicts where available and original otherwise.

In [12]:
# Distribution of utilities/rates within each state
state_totals = agents.groupby("state_abbr")["customers_in_bin_initial"].sum().rename("state_total")

rate_dist = (
    agents.groupby(["state_abbr", "utility_name", "tariff_name"], dropna=False)["customers_in_bin_initial"]
    .sum()
    .reset_index()
    .merge(state_totals, on="state_abbr")
)
rate_dist["pct_of_state"] = (100 * rate_dist["customers_in_bin_initial"] / rate_dist["state_total"]).round(1)
rate_dist = rate_dist.sort_values(["state_abbr", "pct_of_state"], ascending=[True, False])

out_path = os.path.join(DATA_DIR, "rate_distribution_by_state.csv")
rate_dist[["state_abbr", "utility_name", "tariff_name", "customers_in_bin_initial", "pct_of_state"]].to_csv(out_path, index=False)
print(f"Saved to {out_path}")
rate_dist[["state_abbr", "utility_name", "tariff_name", "customers_in_bin_initial", "pct_of_state"]]

Saved to ../../../data/rate_distribution_by_state.csv


,state_abbr,utility_name,tariff_name,customers_in_bin_initial,pct_of_state
0,AL,Alabama Power Co,Family Dwelling Service,1801230.318315,90.6
1,AL,Black Warrior Elec Member Corp,City Limits- Residential,36358.576162,1.8
10,AL,"Wiregrass Electric Coop, Inc",Ancillary 1,34383.57104,1.7
2,AL,Central Alabama Electric Coop,Residential Service,32394.590729,1.6
9,AL,"Tombigbee Electric Coop, Inc",Residential,19612.256088,1.0
...,...,...,...,...,...
1064,WY,"Wheatland Rural Elec Assn, Inc",Residential Service,1942.363821,0.9
1052,WY,"Bridger Valley Elec Assn, Inc (Utah)",Small General Service - Single Phase,1809.405597,0.8
1054,WY,"Butte Electric Coop, Inc",Residential- Demand,1472.051931,0.7
1060,WY,"Niobrara Electric Assn, Inc",Three Phase Time of Use- Residential,1096.045265,0.5


In [10]:
# Use updated tariff_dict where available, else original
def best_tariff(row):
    td = row["tariff_dict_updated"]
    if td is not None:
        return td
    return row["tariff_dict_original"]

agents["_best_rate"] = agents.apply(
    lambda row: effective_rate(best_tariff(row), agent_kwh(row)), axis=1
)

state_avg = (
    agents.groupby("state_abbr")
    .apply(lambda g: pd.Series({
        "old_rate":     wavg(g, "_old_rate"),
        "updated_rate": wavg(g, "_best_rate"),
        "n_agents": len(g),
        "total_households": g["customers_in_bin_initial"].sum(),
    }), include_groups=False)
    .reset_index()
    .sort_values("state_abbr")
)

state_avg["change"] = state_avg.apply(
    lambda r: f"+{(r.updated_rate / r.old_rate - 1)*100:.0f}%"
    if pd.notna(r.updated_rate) and pd.notna(r.old_rate) and r.old_rate > 0 else "—",
    axis=1
)
state_avg["old_avg_$/kWh"]     = state_avg["old_rate"].apply(lambda x: f"${x:.4f}" if pd.notna(x) else "—")
state_avg["updated_avg_$/kWh"] = state_avg["updated_rate"].apply(lambda x: f"${x:.4f}" if pd.notna(x) else "—")
state_avg["total_households"]  = state_avg["total_households"].round(0).astype(int)

print(state_avg[["state_abbr","old_avg_$/kWh","updated_avg_$/kWh","change","total_households"]].to_string(index=False))

# Save
state_avg_path = os.path.join(DATA_DIR, "state_avg_price_updated.csv")
state_avg[["state_abbr","old_avg_$/kWh","updated_avg_$/kWh","change","total_households"]].to_csv(state_avg_path, index=False)
print(f"\nSaved to {state_avg_path}")

state_abbr old_avg_$/kWh updated_avg_$/kWh change  total_households
        AL       $0.1050           $0.1391   +32%           1987606
        AR       $0.0738           $0.0738    +0%           1186792
        AZ       $0.1299           $0.1643   +26%           2687884
        CA       $0.2614           $0.3549   +36%          13125723
        CO       $0.1021           $0.1269   +24%           1974603
        CT       $0.1746           $0.2938   +68%           1174973
        DC       $0.2030           $0.2040    +0%            302334
        DE       $0.1015           $0.1217   +20%            371604
        FL       $0.0986           $0.1255   +27%           9595398
        GA       $0.0720           $0.0927   +29%           3763509
        IA       $0.0841           $0.1143   +36%           1198314
        ID       $0.0871           $0.1092   +25%            608297
        IL       $0.1010           $0.1528   +51%           4502253
        IN       $0.1109           $0.1310   +18